In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def hybrid_ranking(df):
    """
    Normalizes features and computes a weighted final score for ranking.
    """
    df_copy = df.copy()
    
    if df_copy.empty:
        return df_copy
    
    # Columns to normalize
    norm_cols = [
        'compatibility_score', 'queue_probability', 'high_utilization_probability',
        'reliability_score', 'predicted_duration_mins', 'estimated_total_cost'
    ]
    
    # Lower-is-better columns (to be inverted)
    invert_cols = ['queue_probability', 'high_utilization_probability', 'predicted_duration_mins', 'estimated_total_cost']
    
    scaler = MinMaxScaler()
    df_copy[norm_cols] = scaler.fit_transform(df_copy[norm_cols])
    
    # Invert designated columns (1 - normalized_value)
    for col in invert_cols:
        df_copy[col] = 1 - df_copy[col]
    
    # Final Score Weighting
    df_copy['final_score'] = (
        0.20 * df_copy['compatibility_score'] +
        0.20 * df_copy['queue_probability'] +
        0.10 * df_copy['high_utilization_probability'] +
        0.20 * df_copy['reliability_score'] +
        0.10 * df_copy['predicted_duration_mins'] +
        0.20 * df_copy['estimated_total_cost']
    )
    
    # Sort and return top 5
    ranked_df = df_copy.sort_values(by='final_score', ascending=False).head(5)
    
    return ranked_df
